In [3]:
# 💻🔄 Restart runtime to reload environment
import os
os._exit(0)

: 

In [2]:
# 🧹 Clean up anything that could interfere
!pip uninstall -y langchain langchain-core langchain-community langchain-openai langchain-huggingface langchain-text-splitters openai gradio transformers faiss-cpu sentence-transformers 2>/dev/null || true

# Upgrade tooling
!pip install -q --upgrade pip setuptools wheel
!pip install -q python-dotenv

# Compatible LangChain set (no langchain-huggingface to avoid version conflicts)
!pip install -q langchain-core==0.2.43
!pip install -q langchain==0.2.10
!pip install -q langchain-community==0.2.10
!pip install -q langchain-openai==0.1.13
!pip install -q langchain-text-splitters==0.2.4
!pip install -q openai==1.58.1
!pip install -q faiss-cpu==1.8.0.post1
!pip install -q gradio==4.44.0 jedi==0.19.1
!pip install -q pypdf pdfplumber

# ✅ Verify imports (FAISS + OpenAI only; no HuggingFace to avoid conflicts)
from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

print("✅ Everything installed and imported successfully!")

Found existing installation: langchain 0.2.10
Uninstalling langchain-0.2.10:
  Successfully uninstalled langchain-0.2.10
Found existing installation: langchain-core 0.2.43
Uninstalling langchain-core-0.2.43:
  Successfully uninstalled langchain-core-0.2.43
Found existing installation: langchain-community 0.2.10
Uninstalling langchain-community-0.2.10:
  Successfully uninstalled langchain-community-0.2.10
Found existing installation: langchain-openai 0.1.13
Uninstalling langchain-openai-0.1.13:
  Successfully uninstalled langchain-openai-0.1.13
Found existing installation: langchain-text-splitters 0.2.4
Uninstalling langchain-text-splitters-0.2.4:
  Successfully uninstalled langchain-text-splitters-0.2.4
Found existing installation: openai 1.58.1
Uninstalling openai-1.58.1:
  Successfully uninstalled openai-1.58.1
Found existing installation: gradio 4.44.0
Uninstalling gradio-4.44.0:
  Successfully uninstalled gradio-4.44.0
Found existing installation: faiss-cpu 1.8.0.post1
Uninstalling

In [3]:
# Import the warnings module and suppress any warnings that might appear during execution
import warnings
warnings.filterwarnings("ignore")  # Ignore warnings to keep the output clean and focused

### **Step 2. Imports and Configuration**


In [4]:
import os
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.text_splitter import CharacterTextSplitter
from langchain_community.vectorstores import FAISS


In [5]:
import os
from dotenv import load_dotenv

# Load .env from the current directory (or parent, if you run from a subfolder)
load_dotenv()

# Read keys (None or "" if missing)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

# Set in environment so LangChain / OpenAI / Tavily use them
if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
    print(f"OPENAI_API_KEY: {OPENAI_API_KEY[:10]} ...")
else:
    print("OPENAI_API_KEY not set in .env")

if TAVILY_API_KEY:
    os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY
    print("TAVILY_API_KEY is set.")
else:
    print("TAVILY_API_KEY not set.")

OPENAI_API_KEY: sk-proj-Wu ...
TAVILY_API_KEY is set.


In [ ]:
# from google.colab import userdata

# OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
# print(f"OPENAI_API_KEY: {OPENAI_API_KEY[:7]} .....")

# TAVILY_API_KEY = userdata.get('TAVILY_API_KEY')
# print(f"TAVILY_API_KEY: {TAVILY_API_KEY[:7]} .....")

# LANGCHAIN_API_KEY = userdata.get('LANGSMITH_API_KEY')
# print(f"LANGCHAIN_API_KEY: {LANGCHAIN_API_KEY[:7]} .....")

In [ ]:
# from langchain_community.document_loaders import PyPDFDirectoryLoader
# from langchain_community.document_loaders import PyPDFLoader
# loader = PyPDFDirectoryLoader("/content/Manuals")
# docs = loader.load()
# # Each 'doc' in docs represents one page of the PDF
# print(f"Loaded {len(docs)} pages.")

###  **FAISS + pdfplumber** (preserves &lt;, &gt;, and other symbols)

PyPDF can drop or mangle characters like `<` and `>` in manuals. Use **pdfplumber** for extraction and **Chroma** for storage so the LLM receives exact text. Run the cells below instead of (or after) the FAISS cells to build this knowledge base.

In [6]:
# Load PDFs with pdfplumber (preserves <, >, and other symbols that PyPDF often drops)
import os
import re
from pathlib import Path
from langchain_core.documents import Document
import pdfplumber

def load_pdfs_with_pdfplumber(directory=".", pattern="*.pdf"):
    """Load all PDFs from directory; preserves special characters like < and >."""
    docs = []
    path = Path(directory)
    for pdf_path in sorted(path.glob(pattern)): # Iterate through all PDF files in the specified directory.
        with pdfplumber.open(pdf_path) as pdf:
            for i, page in enumerate(pdf.pages): # Process each page of the PDF.
                text = page.extract_text()
                if not text or not text.strip():
                    continue
                # Minimal clean: only collapse multiple newlines/spaces; do NOT remove < or >.
                # This preserves crucial formatting and symbols often found in technical manuals.
                text = re.sub(r"\n+", "\n", text)
                text = re.sub(r" +", " ", text)
                text = text.strip()
                docs.append(Document(
                    page_content=text,
                    metadata={"source": str(pdf_path.name), "page": i + 1} # Store source file and page number as metadata.
                ))
    return docs

# Load all Samsung manual PDFs from current folder using pdfplumber.
# These 'manual_docs' will then be processed and stored in the Chroma vector database.
manual_docs = load_pdfs_with_pdfplumber(".", "*.pdf")
print(f"Loaded {len(manual_docs)} pages from PDFs. Sample (first 400 chars):")
print(manual_docs[0].page_content[:400] if manual_docs else "No documents.")

Loaded 464 pages from PDFs. Sample (first 400 chars):
Refrigerator
User manual
RF18A5104SR
Free Standing Appliance / THAI SAMSUNG ELECTRONICS CO.,LTD


### **FAISS Vector Store: Document Chunking, Embedding, and Storage**

This section details the workflow for processing documents, creating their vector embeddings, and storing them in FAISS. After robust text extraction using `pdfplumber` (as discussed in the alternative approach), the documents are split into manageable chunks, transformed into numerical representations (embeddings) using OpenAI's embedding model, and then persisted in FAISS for efficient retrieval. This forms the knowledge base for our RAG system, allowing for semantic search against user queries.

In [7]:

# Split and store in FAISS (exact text is stored and returned to the LLM)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

FAISS_INDEX_DIR = "faiss_manuals"
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
    separators=["\n\n", "\n", ". ", " ", ""],  # avoid splitting on < or >
)
chunks = text_splitter.split_documents(manual_docs)
embeddings = OpenAIEmbeddings()

# FAISS: build index and save to disk (preserves <, >, etc. in stored text)
vectorStore = FAISS.from_documents(chunks, embeddings)
vectorStore.save_local(FAISS_INDEX_DIR)
print(f"Stored {len(chunks)} chunks in FAISS at '{FAISS_INDEX_DIR}'.")

Stored 849 chunks in FAISS at 'faiss_manuals'.


In [10]:
# Optional: load existing FAISS index (run this instead of re-running the two cells above)
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
embeddings = OpenAIEmbeddings()
vectorStore = FAISS.load_local("faiss_manuals", embeddings, allow_dangerous_deserialization=True)

In [12]:
# Perform a similarity search on the vector store (which is our ChromaDB instance if the alternative path was followed).
# This simulates how the RAG system will retrieve relevant documents for a user's question.
# The query "How to clean icemaker?" is used to find semantically similar chunks within ChromaDB.
# The 'docs' variable will contain a list of retrieved Document objects from ChromaDB.
docs = vectorStore.similarity_search("How to start the icemaker?")
# Display the retrieved documents to inspect their content and metadata.
docs


[Document(metadata={'source': 'RF23DB965012AP_OID74582-04_T-TYPE_RF9000D_EN_MES_260116.pdf', 'page': 59, 'start_index': 0}, page_content='Auto Ice maker (applicable models only)\nThe refrigerator has a built-in ice maker that automatically dispenses ice.\n• The overall design and/or accessories may differ with the model.\nWARNING\nChoking hazard: Ice may cause choking in\nsmall children.\nCAUTION\nAfter using the ice scoop, do not leave it in\nthe ice bucket.\nFor first-time use\n• Let the ice maker make ice for 1-2 days.\n• Discard the first 1-2 buckets of ice to remove impurities in the water supply system.\nC ubed ice/ice bites and sphere ice maker (Applicable models only)\nA. Cubed ice * or ice bites *\nA B B. Sphere ice *\nC. Sphere ice maker pad *\nC\nEnglish 59'),
 Document(metadata={'source': 'RF23DB965012ED_OID74582-04_T-TYPE_RF9000D_EN_MES_260116.pdf', 'page': 59, 'start_index': 0}, page_content='Auto Ice maker (applicable models only)\nThe refrigerator has a built-in ice mak

In [13]:
import os
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA
from langchain_openai import ChatOpenAI

# GPT-4o mini: fast, cost-effective, good at following "answer only from context"
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [15]:
question = "How to clean the icemaker?"

template = """You are a Samsung technician support assistant. Answer only using the provided excerpts from the official manuals.
Do not guess or provide inaccurate information. If the answer is not found in the context, say you don't know.
Context: {context}
Question: {question}
Answer:"""

# System + user ip + Context

QA_CHAIN_PROMPT = PromptTemplate.from_template(template)

# Create Retrieval QA Chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorStore.as_retriever(search_type="similarity", k=2),
    chain_type_kwargs={"prompt": QA_CHAIN_PROMPT}
)

# Run the chain with the user query
result = qa_chain.invoke({"query": question})
answer = result['result']

In [16]:
print("answer is\n", answer)

answer is
 Clean the Tray Ice Maker once every one to two weeks. Clean the Tray Ice Maker before using it for the first time. After cleaning the Ice Tray Maker, completely dry all parts before use.


### **OpenAI Functions Agent (tool-calling)**

The agent uses the LLM with **function calling**: it can call tools (e.g. search the manual) when needed. Built with `create_openai_functions_agent` and `AgentExecutor`. Requires `vectorStore` and `llm` from the cells above.

In [ ]:
# --- Libraries for OpenAI Functions Agent ---
from langchain.agents import create_openai_functions_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import create_retriever_tool, tool
from langchain_core.messages import HumanMessage, AIMessage
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.chat_message_histories import ChatMessageHistory

# In-memory store for chat history per session (session_id -> ChatMessageHistory)
store = {}
current_session_id = "default"

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

@tool
def get_chat_history() -> str:
    """Returns the recent conversation history (previous questions and answers in this chat). Use when the user refers to something they asked earlier or when you need context from the conversation."""
    if current_session_id not in store:
        return "No previous messages in this conversation."
    hist = store[current_session_id]
    parts = []
    for m in hist.messages:
        if isinstance(m, HumanMessage):
            parts.append(f"User: {m.content}")
        elif isinstance(m, AIMessage):
            parts.append(f"Assistant: {m.content}")
    return "\n".join(parts[-20:]) if parts else "No previous messages."

# Tool: search the Samsung manual (uses existing vectorStore)
# Run the FAISS/vectorStore and llm cells above first.
retriever = vectorStore.as_retriever(search_type="similarity", k=4)
manual_search_tool = create_retriever_tool(
    retriever,
    name="manual_search",
    description="Search the Samsung refrigerator manual for cleaning, error codes, defrost, parts, and usage.",
)
search = TavilySearchResults()
tools = [manual_search_tool, search, get_chat_history]

# Prompt includes chat_history so the agent sees previous turns; use get_chat_history tool when you need to recall conversation
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a Samsung technician assistant. Use manual_search for the refrigerator manual (cleaning, error codes, defrost, parts) and rephrase the answer whereever required. Use tavily_search when you need general or web info. Use get_chat_history when the user refers to something they asked earlier or when you need conversation context. Prefer the manual when relevant; otherwise use web search."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# Build the agent and executor (verbose=True shows tool calls; return_intermediate_steps=True returns steps in the result)
agent = create_openai_functions_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, return_intermediate_steps=True)

print('Agent ready. Use: agent_executor.invoke({"input": "How to make ice"})')

Agent ready. Use: agent_executor.invoke({"input": "How to make ice"})


In [22]:
# Use memory: pass chat_history and update store after the reply
current_session_id = "default"
history = get_session_history(current_session_id)
chat_history = history.messages

result = agent_executor.invoke({
    "input": "How to defrost the icemaker?",
    "chat_history": chat_history,
})
history.add_user_message("How to defrost the icemaker?")
history.add_ai_message(result["output"])

print(result["output"])



> Entering new AgentExecutor chain...

Invoking: `manual_search` with `{'query': 'defrost icemaker'}`


when you open or close the door.
• When the refrigerator recovers power after a power failure, the ice bucket may
contain a mix of melted and jammed ice cubes, which can prevent the ice maker from
working properly. To prevent this, make sure to empty the ice bucket before using the
refrigerator.
• Do not put fingers or any objects into the ice maker. This can cause physical injury or
property damage.
• Ice stored in the freezer for a long time gets smaller and then forms large ice chunks
caused by sublimation. Therefore, if you are not planning to use the ice maker for a long
time, turn it off and empty the ice bucket.
• Use the new hose-sets supplied with the appliance only. Do not re-use an old hose set.
62 English

when you open or close the door.
• When the refrigerator recovers power after a power failure, the ice bucket may
contain a mix of melted and jammed ice cubes, which 

In [ ]:
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA
from langchain.schema import AIMessage, HumanMessage
import gradio as gr
import traceback

# Toggle this flag to True to enable debug logs
DEBUG = True

def predict(message, history):
    try:
        if DEBUG: print(f"\n📥 Received message: {message}")
        history_langchain_format = []

        # Convert chat history into LangChain format
        for human, ai in history:
            history_langchain_format.append(HumanMessage(content=human))
            history_langchain_format.append(AIMessage(content=ai))

        # Define Prompt
        template = """You are a Samsung technician support assistant. Answer only using the provided excerpts from the official manuals.
        Do not guess or provide inaccurate information. If the answer is not found in the context, say you don’t know.
        Context: {context}
        Question: {question}
        Answer:"""
        QA_CHAIN_PROMPT = PromptTemplate.from_template(template)

        # Manual-only RAG (no Tavily/agent)
        result = qa_chain.invoke({"query": message})
        answer = result['result']
        if DEBUG: print(f"✅ Answer:\n{answer}")
        return answer

    except Exception as e:
        error_trace = traceback.format_exc()
        if DEBUG: print(f"❌ Exception occurred:\n{error_trace}")
        return "⚠️ Sorry, an internal error occurred. Please try again later."

# Gradio Chatbot UI
gr.ChatInterface(
    fn=predict,
    chatbot=gr.Chatbot(height=300),
    textbox=gr.Textbox(placeholder="Ask about Samsung refrigerator manuals (e.g. cleaning, error codes, defrost)", container=False, scale=7),
    title="Samsung Technician Manual Q&A",
    theme="soft",
    examples=[
        "How do I clean the icemaker?",
        "What does error code 22E mean?",
        "How does the defrost cycle work?"
    ],
    retry_btn=None,
    undo_btn="Delete Previous",
    clear_btn="Clear",
).launch(share=True)

In [ ]:
# Run the agent with memory (FAISS + Tavily + chat history tool)
current_session_id = "default"
history = get_session_history(current_session_id)
user_input = "How do I clean the icemaker?"

result = agent_executor.invoke({
    "input": user_input,
    "chat_history": history.messages,
})
history.add_user_message(user_input)
history.add_ai_message(result["output"])

print(result["output"])